In [2]:
import numpy as np
import sklearn
from sklearn.linear_model import LogisticRegression

# You are allowed to import any submodules of numpy or sklearn e.g. np.random.randint for random initialization or sklearn.metrics.accuracy_score to calculate accuracy of a learnt model
# You are not allowed to use other libraries such as scipy, keras, tensorflow etc

# SUBMIT YOUR CODE AS A SINGLE PYTHON (.PY) FILE INSIDE A ZIP ARCHIVE
# THE NAME OF THE PYTHON FILE MUST BE submit.py

# DO NOT CHANGE THE NAME OF THE METHODS my_latent, my_latent_updated etc BELOW
# THESE WILL BE INVOKED BY THE EVALUATION SCRIPT. CHANGING THESE NAMES WILL CAUSE EVALUATION FAILURE

# You may define any new functions, variables, classes here

# ---------------------------------------------------------------------------
# Hyperparameters -- retune these against your own held-out validation split.
# Trimmed down from the first draft because the E/M loop was burning its full
# iteration budget every restart (see comments below on warm-starting).
# ---------------------------------------------------------------------------
N_RESTARTS = 3        # number of random re-initializations of the latent z's
MAX_ITERS = 15         # max E/M alternations per restart
REG_C = 1.0            # inverse regularization strength for LogisticRegression
INNER_MAX_ITER = 200   # solver budget for fits *during* the alternating loop
FINAL_MAX_ITER = 2000  # solver budget for the one high-precision refit at the end
STOP_FRAC = 0.001      # treat the loop as converged once <=0.1% of z's flip


def _insert_bit(C, z_col):
    """
    Re-insert a guessed middle bit into a 16-bit challenge to recover a
    17-bit challenge: I(c, z) = [c[:8], z, c[8:]] (matches the handout).
    C     : (n, 16) binary challenge matrix
    z_col : (n,) binary vector, the bit to place at position index 8
    returns (n, 17) binary challenge matrix
    """
    return np.hstack([C[:, :8], z_col.reshape(-1, 1), C[:, 8:]])


def _phi(C):
    """
    Standard arbiter-PUF challenge -> feature transform (as discussed in
    class):  phi_i = prod_{j=i}^{k-1} (1 - 2 * c_j),  i = 0, ..., k-1
    i.e. the suffix product of +-1 signs. Maps {0,1}^k -> {-1,+1}^k.

    NOTE: if your own lecture notes / earlier assignment define phi
    differently, swap that exact transform in here -- everything below
    only depends on _phi and _insert_bit.
    """
    s = 1 - 2 * C  # 0 -> +1, 1 -> -1
    return np.cumprod(s[:, ::-1], axis=1)[:, ::-1]


def _score(Phi, w, b):
    """Linear score w^T phi + b, vectorized over rows of Phi."""
    return Phi @ w + b


def _log_sigmoid(t):
    """Numerically stable log(sigmoid(t))."""
    return -np.logaddexp(0, -t)


def _new_warm_clf(max_iter):
    """
    A LogisticRegression with warm_start=True: calling .fit() on the SAME
    instance again re-uses its previous coefficients as the initial point,
    instead of restarting from zero. Since consecutive E/M iterations only
    change a handful of labels/features, this collapses each refit from
    ~15-30 solver iterations down to just a few.
    """
    return LogisticRegression(C=REG_C, tol=1e-4, max_iter=max_iter, warm_start=True)


def _fit_logreg(clf, Phi, labels):
    """Fit (or warm-refit) clf and return (w, b)."""
    clf.fit(Phi, labels)
    return clf.coef_.ravel(), clf.intercept_[0]


################################
# Non Editable Region Starting #
################################
def my_latent( X, y ):
################################
#  Non Editable Region Ending  #
################################

    # Hard-EM / winner-take-all alternating optimization:
    #   E-step: fix (w,b); set each z_i to whichever of {0,1} makes the
    #           observed r_i more likely under the current model
    #   M-step: fix {z_i}; refit (w,b) as an ordinary logistic regression
    #           on the reconstructed 17-bit features
    # Multiple random restarts guard against bad initializations since the
    # joint problem is non-convex.

    n = X.shape[0]
    rng = np.random.default_rng(0)

    # These only depend on X (not on the current z or model), so precompute once
    Phi17_0 = _phi(_insert_bit(X, np.zeros(n, dtype=int)))
    Phi17_1 = _phi(_insert_bit(X, np.ones(n, dtype=int)))
    signed_y = 2 * y - 1

    stop_count = max(1, int(STOP_FRAC * n))
    best_w = best_b = best_z = None
    best_acc = -1.0

    for _ in range(N_RESTARTS):
        z = rng.integers(0, 2, size=n)
        clf = _new_warm_clf(INNER_MAX_ITER)

        for _ in range(MAX_ITERS):
            Phi17 = np.where(z[:, None] == 1, Phi17_1, Phi17_0)
            w, b = _fit_logreg(clf, Phi17, y)

            score0 = _score(Phi17_0, w, b)
            score1 = _score(Phi17_1, w, b)
            z_new = (signed_y * score1 > signed_y * score0).astype(int)

            changed = np.sum(z_new != z)
            z = z_new
            if changed <= stop_count:
                break

        # one high-precision refit on the converged z before scoring
        Phi17 = np.where(z[:, None] == 1, Phi17_1, Phi17_0)
        clf.max_iter = FINAL_MAX_ITER
        w, b = _fit_logreg(clf, Phi17, y)
        acc = np.mean((_score(Phi17, w, b) > 0).astype(int) == y)

        if acc > best_acc:
            best_acc, best_w, best_b, best_z = acc, w, b, z.astype(bool)

    return best_w, best_b, best_z

################################
# Non Editable Region Starting #
################################
def my_latent_updated( X, y ):
################################
#  Non Editable Region Ending  #
################################

    # Same hard-EM skeleton as my_latent, but now z_i has a learned prior
    # P[z_i | c_i, u, a] = sigma((2z_i-1)(u.phi(c_i)+a)) instead of a flat
    # 0.5 prior, so both (w,b) and (u,a) are refit every M-step, and the
    # E-step scores z_i using both models jointly.

    n = X.shape[0]
    rng = np.random.default_rng(1)

    Phi16 = _phi(X)
    Phi17_0 = _phi(_insert_bit(X, np.zeros(n, dtype=int)))
    Phi17_1 = _phi(_insert_bit(X, np.ones(n, dtype=int)))
    signed_y = 2 * y - 1

    stop_count = max(1, int(STOP_FRAC * n))
    best_w = best_b = best_u = best_a = None
    best_ll = -np.inf

    for _ in range(N_RESTARTS):
        z = rng.integers(0, 2, size=n)
        clf_w = _new_warm_clf(INNER_MAX_ITER)
        clf_u = _new_warm_clf(INNER_MAX_ITER)

        for _ in range(MAX_ITERS):
            Phi17 = np.where(z[:, None] == 1, Phi17_1, Phi17_0)
            w, b = _fit_logreg(clf_w, Phi17, y)
            u, a = _fit_logreg(clf_u, Phi16, z)

            r0 = _log_sigmoid(signed_y * _score(Phi17_0, w, b))
            r1 = _log_sigmoid(signed_y * _score(Phi17_1, w, b))
            zscore = _score(Phi16, u, a)
            p0 = _log_sigmoid(-zscore)
            p1 = _log_sigmoid(zscore)

            z_new = ((r1 + p1) > (r0 + p0)).astype(int)
            changed = np.sum(z_new != z)
            z = z_new
            if changed <= stop_count:
                break

        # one high-precision refit on the converged z before scoring
        Phi17 = np.where(z[:, None] == 1, Phi17_1, Phi17_0)
        clf_w.max_iter = FINAL_MAX_ITER
        clf_u.max_iter = FINAL_MAX_ITER
        w, b = _fit_logreg(clf_w, Phi17, y)
        u, a = _fit_logreg(clf_u, Phi16, z)

        ll = (np.sum(_log_sigmoid(signed_y * _score(Phi17, w, b))) +
              np.sum(_log_sigmoid((2 * z - 1) * _score(Phi16, u, a))))

        if ll > best_ll:
            best_ll, best_w, best_b, best_u, best_a = ll, w, b, u, a

    return best_w, best_b, best_u, best_a